# CS781 Phase 2 - L-NER - ADAPTATION TECHNIQUES - model D (Gated DeltaNet + MLA)
Compares adaptation techniques on this backbone (full fine-tuning and LoRMA+ r16 were already run in the FINAL notebooks):
**LoRA**, **Houlsby adapters**, **partial fine-tuning** (top 4 blocks), **LoRMA+ with rank 32**, **regularised full fine-tuning** (dropout 0.3, wd 0.1), **norm-only** (dev only).
Two jobs in parallel (one per T4). For each technique: a dev run (`<m>_<tech>`: 49 train / 21 dev docs, 10 epochs, best epoch) and an all-data run (`<m>_<tech>_all`: 70 docs, 6 epochs).
Other settings as before: BiLSTM head, `piece` tokenization, sqrt class weights, window 512, BIO-Viterbi.

**Attach as inputs:** code dataset, `cs781-legal-data`, the Phase-1 checkpoint of model **D** only. Internet ON, secret `HF_TOKEN`, GPU T4 x2.
Save Version -> Save & Run All (about 1-1.5 h). Afterwards download from the Output tab: `bundle_d_tech.zip` (for the ensemble) and the folder `submissions/` (ready-to-submit zips named `D_<technique>_<dev49|all70>.zip`).

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading
WORK = "/kaggle/working/p2code"
P2DATA = "/kaggle/working/p2data"
RUNS = "/kaggle/working/runs"

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("ner_train.py"), WORK)
os.chdir(WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", WORK, sorted(f for f in os.listdir(WORK) if f.endswith(".py")))
print("tokenizer root:", TOK_ROOT, os.listdir(TOK_ROOT))

# ---- Phase-2 data from Hugging Face (Train folds + labels; Test fold3 is read only by ner_train.py for prediction)
os.makedirs(P2DATA, exist_ok=True)
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
for f in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json", "Train/fold2-gold.json",
          "Train/labels.txt", "Test/fold3-text.json"]:
    p = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{f}", repo_type="dataset", token=tok_)
    shutil.copy(p, os.path.join(P2DATA, os.path.basename(f)))
TEST = os.path.join(P2DATA, "fold3-text.json")
print("p2 data:", os.listdir(P2DATA), "test docs:", len(json.load(open(TEST))))

# ---- Phase-1 checkpoints: final.pt per model letter (continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
os.makedirs("/kaggle/working/logs", exist_ok=True)

def run_parallel(jobs, timeout=None):
    """jobs: list of (tag, shell_cmd, gpu_index_or_None). Runs them at the same time (one per GPU), streams
    output prefixed with the tag, saves /kaggle/working/logs/<tag>.txt, returns {tag: (exit_code, text)}."""
    procs, outs, lock = {}, {}, threading.Lock()
    def reader(tag, p):
        buf = []
        for line in p.stdout:
            buf.append(line)
            with lock:
                print(f"[{tag}] {line}", end="", flush=True)
        outs[tag] = buf
    threads = []
    for tag, cmd, gpu in jobs:
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        print(f"$ [{tag}] (GPU {gpu}) {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env, start_new_session=True)
        procs[tag] = p
        t = threading.Thread(target=reader, args=(tag, p), daemon=True); t.start(); threads.append(t)
    timer = None
    if timeout:
        def kill():
            print("TIMEOUT - killing", flush=True)
            for p in procs.values():
                try: os.killpg(p.pid, 9)
                except Exception: pass
        timer = threading.Timer(timeout, kill); timer.start()
    for p in procs.values(): p.wait()
    for t in threads: t.join()
    if timer: timer.cancel()
    res = {}
    for tag, p in procs.items():
        txt = "".join(outs.get(tag, []))
        open(f"/kaggle/working/logs/{tag}.txt", "w").write(txt)
        res[tag] = (p.returncode, txt)
        print(f"[{tag}] exit {p.returncode}", flush=True)
    return res

code: /kaggle/working/p2code ['attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_ensemble.py', 'ner_eval.py', 'ner_model.py', 'ner_post.py', 'ner_train.py', 'p1_loader.py', 'peft.py', 'tests_ner.py']
tokenizer root: /kaggle/input/datasets/mrinaldevnath/cs781-legal-data ['tokenizer', 'manifest.json', 'p1_loader.py', 'prep_p1.py', 'legal_bpe_v2.py', 'val', 'train', 'dev', 'dev_prompts.jsonl']


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

p2 data: ['fold3-text.json', 'fold2-text.json', 'fold1-gold.json', 'fold2-gold.json', 'fold1-text.json', 'labels.txt'] test docs: 35
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB



## 1. Tests (CPU, ~2 min)

In [2]:
r = subprocess.run(f"python tests_ner.py --tok_root {TOK_ROOT} --data_dir {P2DATA}", shell=True, capture_output=True, text=True, cwd=WORK)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "tests failed - do not train; see the output above"

ed": 42, "act_ckpt": false, "load": null, "predict_only": false, "cpu": true}
[13:15:52] docs: train 4 (23,470 tokens)  dev 2  test 3  [piece pretok, 0s]
[13:15:52] alignment ceiling (gold -> BIO -> spans) on dev: macro-F1 1.0000
[13:15:52] backbone decoder d=32: 1.07M params (trunk); method=lorma wrapped=10 trainable backbone 0.002M + head 0.607M (36.35% of 1.67M)
[13:15:52] tag weights: O:1.0 B-APP:20.0 I-APP:17.3 B-RESP:20.0 I-RESP:17.3 B-A.COUNSEL:20.0 I-A.COUNSEL:20.0 B-R.COUNSEL:20.0 I-R.COUNSEL:20.0 B-JUDGE:20.0 I-JUDGE:15.2 B-WIT:20.0 I-WIT:20.0 B-AUTH:11.4 I-AUTH:9.5 B-COURT:20.0 I-COURT:20.0 B-STAT:20.0 I-STAT:14.6 B-PREC:20.0 I-PREC:7.4 B-DATE:12.3 I-DATE:5.8 B-CASENO:20.0 I-CASENO:8.2
[13:15:52] lr backbone 0.0005  head 0.001; ~93 steps/epoch x 2 epochs; warmup 11; window 64; centrality center
[13:15:57] epoch 1 loss 2.1982  5,927 tok/s  5s  mem 0.0G  | dev macro-F1 0.0000 (P 0.000 R 0.000) micro 0.0000  *best*
[13:16:02] epoch 2 loss 1.6589  6,228 tok/s  4s  mem 0.0G  | de

## 2. Training

In [3]:
MODEL = "d"
assert MODEL in CKPT, f"no Phase-1 checkpoint for model {MODEL} found under /kaggle/input - attach it"
COMMON = f"--tok_root {TOK_ROOT} --data_dir {P2DATA} --test_file {TEST}"
BASE = "--head bilstm --patience 10"
DEV = "--epochs 10 --dev_frac 0.3"          # dev run: 49 train / 21 dev docs, best epoch picked on dev
ALL = "--epochs 6 --all_data"               # final run: all 70 docs, fixed 6 epochs (dev curves peak at epochs 4-6, then overfit)
# technique -> extra flags.  All use the same data, head, tokenization and decoding; only the adaptation differs.
TECH = {
    "lora":    "--method lora --rank 16 --alpha 32",                 # LoRA r16 on attention + FFN linears
    "adapter": "--method adapter --bottleneck 64",                   # Houlsby bottleneck adapters
    "partial": "--method partial --top_k 4",                         # fine-tune only the top 4 blocks (+ final norm)
    "lorma32": "--method lorma --rank 32 --alpha 64",                # LoRMA+ with doubled rank (r16 was the round-1 setting)
    "fullreg": "--method full --dropout 0.3 --wd 0.1",               # full fine-tuning + stronger regularisation (head dropout 0.3, wd 0.1)
    "norm":    "--method norm",                                      # train only the normalisation scales (BitFit-style); dev run only
}
# one queue per GPU: for each technique a dev run and then the all-data run
QUEUES = [
    [(f"{MODEL}_lora", "lora", DEV), (f"{MODEL}_lora_all", "lora", ALL),
     (f"{MODEL}_adapter", "adapter", DEV), (f"{MODEL}_adapter_all", "adapter", ALL),
     (f"{MODEL}_partial", "partial", DEV), (f"{MODEL}_partial_all", "partial", ALL)],
    [(f"{MODEL}_lorma32", "lorma32", DEV), (f"{MODEL}_lorma32_all", "lorma32", ALL),
     (f"{MODEL}_fullreg", "fullreg", DEV), (f"{MODEL}_fullreg_all", "fullreg", ALL),
     (f"{MODEL}_norm", "norm", DEV)],
]
STATUS, LOCK = {}, threading.Lock()
SKIP_PRINT = ("args {", "tag weights:")

def sh(cmd, gpu, tag, suffix=""):
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=WORK)
    buf = []
    for line in p.stdout:
        buf.append(line)
        if not any(k in line for k in SKIP_PRINT):
            with LOCK:
                print(f"[{tag}] {line}", end="", flush=True)
    p.wait()
    txt = "".join(buf)
    open(f"/kaggle/working/logs/{tag}{suffix}.txt", "w").write(txt)
    return p.returncode, txt

def worker(gpu, queue):
    for name, tech, stage_args in queue:
        out = f"{RUNS}/{name}"
        t0 = time.time()
        if os.path.exists(f"{out}/test_logp.npz"):                       # finished earlier in this session
            STATUS[name] = "ok (already done)"; continue
        cmd = f"python ner_train.py --ckpt {CKPT[MODEL]} {COMMON} --out {out} {BASE} {TECH[tech]} {stage_args}"
        print(f"\n$ [{name}] (GPU {gpu}) {cmd}", flush=True)
        rc, txt = sh(cmd, gpu, name)
        if rc != 0 and "out of memory" in txt.lower():
            print(f"[{name}] OOM -> retrying with --act_ckpt", flush=True)
            shutil.rmtree(out, ignore_errors=True)
            rc, txt = sh(cmd + " --act_ckpt", gpu, name, "_retry")
        STATUS[name] = ("ok (%.1f min)" % ((time.time() - t0) / 60)) if rc == 0 else f"FAILED (exit {rc}) - see /kaggle/working/logs/{name}.txt"
        if rc != 0:
            print(f"[{name}] FAILED exit {rc}", flush=True)

T0 = time.time()
ths = [threading.Thread(target=worker, args=(g, q), daemon=True) for g, q in enumerate(QUEUES)]
for t in ths: t.start()
for t in ths: t.join()
print("\nALL DONE in %.1f min" % ((time.time() - T0) / 60))
for k, v in STATUS.items(): print(" ", k, "->", v)


$ [d_lora] (GPU 0) python ner_train.py --ckpt /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt --tok_root /kaggle/input/datasets/mrinaldevnath/cs781-legal-data --data_dir /kaggle/working/p2data --test_file /kaggle/working/p2data/fold3-text.json --out /kaggle/working/runs/d_lora --head bilstm --patience 10 --method lora --rank 16 --alpha 32 --epochs 10 --dev_frac 0.3

$ [d_lorma32] (GPU 1) python ner_train.py --ckpt /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt --tok_root /kaggle/input/datasets/mrinaldevnath/cs781-legal-data --data_dir /kaggle/working/p2data --test_file /kaggle/working/p2data/fold3-text.json --out /kaggle/working/runs/d_lorma32 --head bilstm --patience 10 --method lorma --rank 32 --alpha 64 --epochs 10 --dev_frac 0.3
[d_lorma32] [13:16:11] docs: train 50 (353,358 tokens)  dev 20  test 35  [piece pretok, 3s]
[d_lora] [13:16:11] docs: train

## 3. Report + bundle + submission files

In [4]:
# ---- report + bundle + READY-TO-SUBMIT files (named <MODEL>_<technique>_<dev49|all70>.zip)
import zipfile
LABS = [l.strip() for l in open(f"{P2DATA}/labels.txt") if l.strip()]
SUB = "/kaggle/working/submissions"
shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
ALLRUNS = [x for q in QUEUES for x in q]
L = [f"MODEL {MODEL.upper()} - adaptation techniques",
     "run | dev macroF1 (21 docs) | greedy | best ep | trainable M | % trainable | train min | peak GB"]
per = {}
for name, tech, sa in ALLRUNS:
    sp = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(sp):
        L.append(f"{name} | {STATUS.get(name, 'not run')}"); continue
    s = json.load(open(sp))
    if "dev_viterbi" in s:
        dv = s["dev_viterbi"]; per[name] = {l: dv["per_label"][l]["f1"] for l in LABS}
        L.append(" | ".join([name, f"{dv['macro_f1']:.4f}", f"{s['dev_greedy']['macro_f1']:.4f}", str(s["best_epoch"]), f"{s['params_trainable'] / 1e6:.2f}",
                             f"{100 * s['params_trainable'] / s['params_total']:.1f}%", f"{s['train_minutes']:.1f}", f"{s['peak_mem_gb']:.1f}"]))
    else:
        L.append(f"{name} | all-data run (no dev score) | {s.get('epochs_run')} epochs | {s.get('params_trainable', 0) / 1e6:.2f}M trainable | {s.get('train_minutes', 0):.1f} min | peak {s.get('peak_mem_gb', 0):.1f} GB")
L.append("\nper-label dev F1")
L.append(f"{'run':16s} " + " ".join(f"{l[:7]:>7s}" for l in LABS))
for t, d in per.items():
    L.append(f"{t:16s} " + " ".join(f"{d[l]:7.2f}" for l in LABS))
with zipfile.ZipFile(f"/kaggle/working/bundle_{MODEL}_tech.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, _, _ in ALLRUNS:
        for f in ("summary.json", "dev_logp.npz", "test_logp.npz", "submission.zip", "train_log.txt"):
            p = f"{RUNS}/{name}/{f}"
            if os.path.exists(p):
                z.write(p, arcname=f"runs/{name}/{f}")
        if os.path.exists(f"{RUNS}/{name}/submission.zip"):
            tech = name[2:].replace("_all", "")
            kind = "all70" if name.endswith("_all") else "dev49"
            shutil.copy(f"{RUNS}/{name}/submission.zip", f"{SUB}/{MODEL.upper()}_{tech}_{kind}.zip")
L.append(f"\nbundle: /kaggle/working/bundle_{MODEL}_tech.zip ({os.path.getsize(f'/kaggle/working/bundle_{MODEL}_tech.zip') / 1e6:.1f} MB)")
L.append("submission files (inside each zip: prediction.json): " + ", ".join(sorted(os.listdir(SUB))))
L.append("status: " + json.dumps(STATUS))
rep = "\n".join(L)
open("/kaggle/working/REPORT.txt", "w").write(rep)
print("=== REPORT ===\n" + rep + "\n=== END REPORT ===")

=== REPORT ===
MODEL D - adaptation techniques
run | dev macroF1 (21 docs) | greedy | best ep | trainable M | % trainable | train min | peak GB
d_lora | 0.5769 | 0.4920 | 6 | 4.94 | 4.1% | 12.0 | 5.9
d_lora_all | all-data run (no dev score) | 6 epochs | 4.94M trainable | 7.9 min | peak 5.9 GB
d_adapter | 0.5887 | 0.4964 | 7 | 4.49 | 3.8% | 10.1 | 4.9
d_adapter_all | all-data run (no dev score) | 6 epochs | 4.49M trainable | 6.6 min | peak 4.9 GB
d_partial | 0.5542 | 0.4541 | 7 | 31.99 | 27.4% | 7.4 | 2.9
d_partial_all | all-data run (no dev score) | 6 epochs | 31.99M trainable | 4.4 min | peak 2.9 GB
d_lorma32 | 0.5818 | 0.4975 | 7 | 8.78 | 7.1% | 12.6 | 6.4
d_lorma32_all | all-data run (no dev score) | 6 epochs | 8.78M trainable | 8.5 min | peak 6.4 GB
d_fullreg | 0.5658 | 0.4724 | 6 | 116.90 | 100.0% | 12.7 | 7.5
d_fullreg_all | all-data run (no dev score) | 6 epochs | 116.90M trainable | 8.6 min | peak 7.5 GB
d_norm | 0.5227 | 0.4121 | 8 | 2.14 | 1.8% | 11.1 | 5.5

per-label dev F1
